In [45]:
pip install nibabel torch torchvision torch-geometric

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [46]:
!pip install scikit-image

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [47]:
import os
import nibabel as nib
import numpy as np
import torch
import torch.nn.functional as F
from skimage.transform import resize
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GCNConv, global_mean_pool

In [48]:
def load_nifti(file_path):
    import nibabel as nib
    img = nib.load(file_path)
    return img.get_fdata()

In [49]:
def preprocess(volume):
    from skimage.transform import resize
    import numpy as np

    volume = resize(volume, (8,8,8))   # if memory issue → change to (8,8,8)
    volume = (volume - np.min(volume)) / (np.max(volume) - np.min(volume))
    return volume

In [50]:
def create_graph(volume):
    import torch
    from torch_geometric.data import Data
    import numpy as np

    patch_size = 2   # 🔥 important

    nodes = []
    edges = []

    x, y, z = volume.shape
    node_id = 0
    index_map = {}

    # Create nodes (patches)
    for i in range(0, x, patch_size):
        for j in range(0, y, patch_size):
            for k in range(0, z, patch_size):

                patch = volume[i:i+patch_size, j:j+patch_size, k:k+patch_size]
                mean_val = np.mean(patch)

                nodes.append([mean_val])
                index_map[(i,j,k)] = node_id
                node_id += 1

    # Create edges (connect nearby patches)
    for (i,j,k), idx in index_map.items():
        for dx,dy,dz in [(patch_size,0,0),(0,patch_size,0),(0,0,patch_size)]:
            ni, nj, nk = i+dx, j+dy, k+dz
            if (ni,nj,nk) in index_map:
                nb = index_map[(ni,nj,nk)]
                edges.append([idx, nb])
                edges.append([nb, idx])

    return Data(
        x=torch.tensor(nodes, dtype=torch.float),
        edge_index=torch.tensor(edges, dtype=torch.long).t().contiguous()
    )

In [51]:
def load_dataset(ad_path, cn_path):
    data_list = []

    # AD = 1
    for f in os.listdir(ad_path):
        if f.endswith(".nii") or f.endswith(".nii.gz"):
            v = preprocess(load_nifti(os.path.join(ad_path,f)))
            g = create_graph(v)
            g.y = torch.tensor([1])
            data_list.append(g)

    # CN = 0
    for f in os.listdir(cn_path):
        if f.endswith(".nii") or f.endswith(".nii.gz"):
            v = preprocess(load_nifti(os.path.join(cn_path,f)))
            g = create_graph(v)
            g.y = torch.tensor([0])
            data_list.append(g)

    return data_list

In [53]:
from sklearn.model_selection import train_test_split

train_data, test_data = train_test_split(
    dataset,
    test_size=0.2,
    stratify=[d.y.item() for d in dataset],
    random_state=42
)

print("Train size:", len(train_data))
print("Test size:", len(test_data))

Train size: 24
Test size: 6


In [54]:
from torch_geometric.loader import DataLoader

train_loader = DataLoader(train_data, batch_size=4, shuffle=True)
test_loader = DataLoader(test_data, batch_size=4, shuffle=False)

In [55]:
class GNN(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = GCNConv(1, 32)
        self.conv2 = GCNConv(32, 64)
        self.dropout = torch.nn.Dropout(p=0.3)   # 🔥 added
        self.fc = torch.nn.Linear(64, 2)

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch

        x = F.relu(self.conv1(x, edge_index))
        x = F.relu(self.conv2(x, edge_index))

        x = global_mean_pool(x, batch)

        x = self.dropout(x)   # 🔥 applied

        x = self.fc(x)
        return F.log_softmax(x, dim=1)

In [56]:
model = GNN()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

for epoch in range(20):
    model.train()
    total_loss = 0

    for data in train_loader:
        optimizer.zero_grad()
        out = model(data)
        loss = F.nll_loss(out, data.y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    print("Epoch:", epoch, "Loss:", total_loss)

Epoch: 0 Loss: 4.148017466068268
Epoch: 1 Loss: 4.171774446964264
Epoch: 2 Loss: 4.162288546562195
Epoch: 3 Loss: 4.157193124294281
Epoch: 4 Loss: 4.164113163948059
Epoch: 5 Loss: 4.1725322008132935
Epoch: 6 Loss: 4.181388676166534
Epoch: 7 Loss: 4.169833660125732
Epoch: 8 Loss: 4.137635231018066
Epoch: 9 Loss: 4.1502684354782104
Epoch: 10 Loss: 4.169660568237305
Epoch: 11 Loss: 4.1558786034584045
Epoch: 12 Loss: 4.1312335729599
Epoch: 13 Loss: 4.1414284110069275
Epoch: 14 Loss: 4.143003404140472
Epoch: 15 Loss: 4.1401262283325195
Epoch: 16 Loss: 4.16868793964386
Epoch: 17 Loss: 4.150731384754181
Epoch: 18 Loss: 4.16255646944046
Epoch: 19 Loss: 4.178621470928192


In [68]:
model.eval()
correct = 0

for data in test_loader:
    out = model(data)
    pred = out.argmax(dim=1)
    correct += int((pred == data.y).sum())

print("Test Accuracy:", correct / len(test_data))

RuntimeError: mat1 and mat2 shapes cannot be multiplied (256x1 and 64x32)

In [70]:
import torch.nn as nn

class CNNClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv3d(1, 8, 3, padding=1)
        self.conv2 = nn.Conv3d(8, 16, 3, padding=1)
        self.pool = nn.MaxPool3d(2)
        self.fc1 = nn.Linear(16*2*2*2, 32)
        self.fc2 = nn.Linear(32, 2)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

In [71]:
X = []
y = []

for label, path in [(1, "alzimers/ad"), (0, "alzimers/cn")]:
    for f in os.listdir(path):
        if f.endswith(".nii") or f.endswith(".nii.gz"):
            vol = preprocess(load_nifti(os.path.join(path, f)))
            X.append(volume_to_tensor(vol))
            y.append(label)

X = torch.cat(X, dim=0)
y = torch.tensor(y)

In [59]:
def volume_to_tensor(volume):
    import torch
    return torch.tensor(volume, dtype=torch.float).unsqueeze(0).unsqueeze(0)

In [60]:
def extract_cnn_features(ad_path, cn_path, cnn_model):
    import os
    import torch

    data_list = []
    cnn_model.eval()

    for label, path in [(1, ad_path), (0, cn_path)]:
        for f in os.listdir(path):
            if f.endswith(".nii") or f.endswith(".nii.gz"):
                vol = preprocess(load_nifti(os.path.join(path, f)))
                tensor = volume_to_tensor(vol)

                with torch.no_grad():
                    feat = cnn_model(tensor).squeeze(0)

                data_list.append((feat, label))

    return data_list

In [61]:
from torch_geometric.data import Data
import torch

def create_feature_graph(data_list):
    nodes = []
    labels = []

    for feat, label in data_list:
        nodes.append(feat.numpy())
        labels.append(label)

    x = torch.tensor(nodes, dtype=torch.float)

    # Fully connected graph
    edge_index = []
    for i in range(len(nodes)):
        for j in range(len(nodes)):
            if i != j:
                edge_index.append([i, j])

    edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()

    return Data(x=x, edge_index=edge_index, y=torch.tensor(labels))

In [62]:
import torch
import torch.nn.functional as F
from torch_geometric.nn import GCNConv

class GNN(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = GCNConv(64, 32)
        self.conv2 = GCNConv(32, 16)
        self.fc = torch.nn.Linear(16, 2)

    def forward(self, data):
        x, edge_index = data.x, data.edge_index

        x = F.relu(self.conv1(x, edge_index))
        x = F.relu(self.conv2(x, edge_index))

        x = self.fc(x)
        return F.log_softmax(x, dim=1)

In [63]:
cnn = CNN3D()

data_list = extract_cnn_features("alzimers/ad", "alzimers/cn", cnn)

graph_data = create_feature_graph(data_list)

print(graph_data)

Data(x=[30, 64], edge_index=[2, 870], y=[30])


C:\Users\gowth\AppData\Local\Temp\ipykernel_18684\2214656427.py:12: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\utils\tensor_new.cpp:256.)
  x = torch.tensor(nodes, dtype=torch.float)


In [67]:
model = GNN()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

for epoch in range(30):
    model.train()
    optimizer.zero_grad()

    out = model(graph_data)
    loss = F.nll_loss(out, graph_data.y)

    loss.backward()
    optimizer.step()

    print("Epoch:", epoch, "Loss:", loss.item())

Epoch: 0 Loss: 0.6947202086448669
Epoch: 1 Loss: 0.6943233013153076
Epoch: 2 Loss: 0.6939877271652222
Epoch: 3 Loss: 0.6937164664268494
Epoch: 4 Loss: 0.6935175657272339
Epoch: 5 Loss: 0.6933652758598328
Epoch: 6 Loss: 0.6932566165924072
Epoch: 7 Loss: 0.6931877136230469
Epoch: 8 Loss: 0.6931549310684204
Epoch: 9 Loss: 0.6931470632553101
Epoch: 10 Loss: 0.6931529641151428
Epoch: 11 Loss: 0.6931664347648621
Epoch: 12 Loss: 0.693183958530426
Epoch: 13 Loss: 0.6932017207145691
Epoch: 14 Loss: 0.6932170987129211
Epoch: 15 Loss: 0.693228542804718
Epoch: 16 Loss: 0.693234384059906
Epoch: 17 Loss: 0.6932348012924194
Epoch: 18 Loss: 0.6932303309440613
Epoch: 19 Loss: 0.6932219862937927
Epoch: 20 Loss: 0.6932111382484436
Epoch: 21 Loss: 0.6931989789009094
Epoch: 22 Loss: 0.6931867599487305
Epoch: 23 Loss: 0.6931752562522888
Epoch: 24 Loss: 0.6931655406951904
Epoch: 25 Loss: 0.6931576132774353
Epoch: 26 Loss: 0.6931520700454712
Epoch: 27 Loss: 0.6931487321853638
Epoch: 28 Loss: 0.693147361278533

In [72]:
model = CNNClassifier()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

for epoch in range(30):
    model.train()
    optimizer.zero_grad()

    out = model(X)
    loss = F.cross_entropy(out, y)

    loss.backward()
    optimizer.step()

    print("Epoch:", epoch, "Loss:", loss.item())

Epoch: 0 Loss: 0.6932176351547241
Epoch: 1 Loss: 0.6917053461074829
Epoch: 2 Loss: 0.690679669380188
Epoch: 3 Loss: 0.6895577907562256
Epoch: 4 Loss: 0.6886418461799622
Epoch: 5 Loss: 0.6876575350761414
Epoch: 6 Loss: 0.6864912509918213
Epoch: 7 Loss: 0.6852867603302002
Epoch: 8 Loss: 0.6840499639511108
Epoch: 9 Loss: 0.6827186942100525
Epoch: 10 Loss: 0.6812419295310974
Epoch: 11 Loss: 0.6795770525932312
Epoch: 12 Loss: 0.6777449250221252
Epoch: 13 Loss: 0.6756993532180786
Epoch: 14 Loss: 0.6733545660972595
Epoch: 15 Loss: 0.6707541942596436
Epoch: 16 Loss: 0.6678033471107483
Epoch: 17 Loss: 0.664722204208374
Epoch: 18 Loss: 0.6613790988922119
Epoch: 19 Loss: 0.6576952338218689
Epoch: 20 Loss: 0.6536081433296204
Epoch: 21 Loss: 0.6491241455078125
Epoch: 22 Loss: 0.6442184448242188
Epoch: 23 Loss: 0.6389337778091431
Epoch: 24 Loss: 0.633040189743042
Epoch: 25 Loss: 0.6266096830368042
Epoch: 26 Loss: 0.6197279691696167
Epoch: 27 Loss: 0.6122681498527527
Epoch: 28 Loss: 0.604349911212921

In [66]:
data_list = extract_cnn_features("alzimers/ad", "alzimers/cn", cnn)
graph_data = create_feature_graph(data_list)

In [73]:
model.eval()
pred = model(X).argmax(dim=1)

acc = (pred == y).sum().item() / len(y)
print("Accuracy:", acc)

Accuracy: 0.8


In [74]:
def get_features(model, X):
    model.eval()
    features = []

    with torch.no_grad():
        for i in range(len(X)):
            x = X[i].unsqueeze(0)
            
            # take features BEFORE final layer
            feat = model.pool(F.relu(model.conv1(x)))
            feat = model.pool(F.relu(model.conv2(feat)))
            feat = feat.view(feat.size(0), -1)

            features.append(feat.squeeze(0))

    return torch.stack(features)

In [75]:
features = get_features(model, X)
print(features.shape)   # should be [N, something]

torch.Size([30, 128])


In [76]:
from sklearn.neighbors import kneighbors_graph

A = kneighbors_graph(features.numpy(), n_neighbors=3, mode='connectivity')

edge_index = []
rows, cols = A.nonzero()

for i, j in zip(rows, cols):
    edge_index.append([i, j])

edge_index = torch.tensor(edge_index).t().contiguous()

In [77]:
gnn = GNN()
optimizer = torch.optim.Adam(gnn.parameters(), lr=0.001)

for epoch in range(30):
    gnn.train()
    optimizer.zero_grad()

    out = gnn(graph_data)
    loss = F.nll_loss(out, graph_data.y)

    loss.backward()
    optimizer.step()

    print("Epoch:", epoch, "Loss:", loss.item())

Epoch: 0 Loss: 0.694147527217865
Epoch: 1 Loss: 0.693899929523468
Epoch: 2 Loss: 0.6936991214752197
Epoch: 3 Loss: 0.6935341954231262
Epoch: 4 Loss: 0.6934311985969543
Epoch: 5 Loss: 0.6933417916297913
Epoch: 6 Loss: 0.6932677030563354
Epoch: 7 Loss: 0.6932106614112854
Epoch: 8 Loss: 0.6931743621826172
Epoch: 9 Loss: 0.6931546926498413
Epoch: 10 Loss: 0.6931473016738892
Epoch: 11 Loss: 0.6931504011154175
Epoch: 12 Loss: 0.6931604146957397
Epoch: 13 Loss: 0.6931732892990112
Epoch: 14 Loss: 0.6931852102279663
Epoch: 15 Loss: 0.6931939721107483
Epoch: 16 Loss: 0.6931976079940796
Epoch: 17 Loss: 0.693196713924408
Epoch: 18 Loss: 0.693191647529602
Epoch: 19 Loss: 0.6931837201118469
Epoch: 20 Loss: 0.693174421787262
Epoch: 21 Loss: 0.6931652426719666
Epoch: 22 Loss: 0.6931573152542114
Epoch: 23 Loss: 0.6931514143943787
Epoch: 24 Loss: 0.6931481957435608
Epoch: 25 Loss: 0.6931473016738892
Epoch: 26 Loss: 0.6931483149528503
Epoch: 27 Loss: 0.693150520324707
Epoch: 28 Loss: 0.6931533813476562
E

In [78]:
gnn.eval()
pred = gnn(graph_data).argmax(dim=1)

acc = (pred == graph_data.y).sum().item() / len(y)
print("Final Accuracy:", acc)

Final Accuracy: 0.5
